# Signpost-D: a decision model trained on our own cases

Signpost answers grounded questions well, but on [Typed Decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) it scored **0.4175** (notebook 06), below the majority baseline. Its weakness there is *judgement*: what should happen next, how serious or urgent something is, inferred from structured information.

**Signpost-D** starts from Signpost's weights and is fine-tuned with a new LoRA on **our own decision cases**:
- written by subagents from a plain-English brief (`data/signpost_d/brief.md`), never from the benchmark;
- **soft-labelled** by three independent labellers (cautious, pragmatic, neutral; averaged);
- checked by validators.

| `DATA_MIX` | Trains on | Question it answers |
|---|---|---|
| `"new"` (run 1) | the Signpost-D cases only | can our own decision data teach the skill? |
| `"new+old"` (run 2) | + a fresh sample of Signpost's original routing and question data each epoch | does the old data help, hurt, or only protect routing? |

**Measured:**
1. our **held-out Signpost-D cases** (accuracy against the label's top option, KL, Brier, ECE), before and after training. These choose the epoch and the temperature;
2. **Signpost's original tests** (routing E10, first messages E11, banking, questions E12, the independent sets): how much was forgotten;
3. the **benchmark test split, once**, at the end, as published (T = 1) and with the temperature fitted on our dev cases.

Nothing is uploaded. One T4 is enough.

In [1]:
# Cell 1 — Setup: packages, settings, device, progress log

!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"

import copy, json, math, os, random, shutil, statistics, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch loads)
from collections import Counter
from pathlib import Path

import pandas as pd
import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

DATA_MIX = "new"                      # "new": the Signpost-D cases only | "new+old": plus Signpost's original training data
D_SET = "pilot"                       # data/signpost_d/<D_SET>/final.jsonl
START = "skamalj/signpost"            # the weights we start from (merged encoder + scoring head)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
LR, HEAD_LR = 2e-4, 5e-4
EPOCHS = 6                            # over the new cases; the best epoch on the dev cases is kept
OLD_PER_EPOCH = 3000                  # new+old: Signpost items sampled afresh each epoch (half routing, half questions)
BATCH_NEW, BATCH_ROUTE, BATCH_CHOICE = 4, 4, 16      # BATCH_NEW 8 ran out of memory on a T4 (long decision states)
MAX_LEN = 512
SCORE_DIST_W = 0.5                    # score questions: weight of |expected level - target expected level| / (levels - 1)
DEV_FRACTION = 0.2
SEED = 42
BENCH = "LocalLLaMA/typed-decisions"  # scored once at the end; never used for training or selection
SEP, ROUTE_Q = " [SEP] ", "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("signpost_d_run.log")
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), "restart the kernel and run from the top"
api = HfApi()
REV = {START: api.model_info(START).sha, BENCH: api.dataset_info(BENCH).sha}


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | {DEVICE} | DATA_MIX {DATA_MIX} | "
    f"start {START} @ {REV[START][:8]}")

[06:17:11] torch 2.11.0+cu128 | transformers 5.17.0 | peft 0.21.2 | cuda:0 | DATA_MIX new | start skamalj/signpost @ e76e5c6c


In [2]:
# Cell 2a — Signpost's original training data (used only when DATA_MIX is new+old) and its dev splits

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():                                   # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

domains = {**load("generic/domains.json"), **load("realistic_train/domains.json")}
route_items = [{"task": "routing", "domain": ex["domain"], "messages": ex["messages"], "current": ex["current_agent"],
                "question": ROUTE_Q, "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]}
               for ex in load("generic/conversations.json") + load("realistic_train/conversations.json")]
choice_items = [{"task": it["type"], "domain": it["domain"], "messages": [{"role": "user", "content": it["state"]}], "current": None,
                 "question": it["question"], "options": {o: "" for o in it["options"]}, "answer": it["answer"], "state": it["state"],
                 "levels": list(it["options"])}            # score questions: the levels, low to high
                for it in load("choice_train/items.json")["items"]]

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items}); rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items}); rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route, dev_route = [x for x in route_items if x["domain"] not in dev_domains], [x for x in route_items if x["domain"] in dev_domains]
train_choice, dev_choice = [x for x in choice_items if x["state"] not in dev_texts], [x for x in choice_items if x["state"] in dev_texts]

for name, xs in [("train routing", train_route), ("train questions", train_choice), ("dev routing", dev_route), ("dev questions", dev_choice)]:
    print(f"{name:15s}: {len(xs):5d} | {dict(Counter(x['task'] for x in xs))}")
print(f"dev routing domains (unseen in training): {sorted(dev_domains)}")

# Signpost-D cases: one item per question, with the labellers' averaged distribution as the target
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


d_cases = [json.loads(l) for l in open(DATA_DIR.parent / "signpost_d" / D_SET / "final.jsonl")]
case_ids = sorted(c["id"] for c in d_cases)
random.Random(SEED).shuffle(case_ids)
dev_ids = set(case_ids[: int(len(case_ids) * DEV_FRACTION)])


def d_items(c):
    out = []
    for q in c["questions"]:
        if q["type"] == "score":
            opts = {l["name"]: l["description"] for l in q["levels"]}
        else:
            opts = dict(q["options"])
        out.append({"task": "d_" + q["type"], "id": c["id"], "qid": q["qid"], "domain": c["domain"],
                    "messages": [{"role": "user", "content": state_text(c["state"])}], "current": None,
                    "question": q["question"], "options": opts, "levels": list(opts) if q["type"] == "score" else None,
                    "soft": q["label"]})
    return out


train_d = [it for c in d_cases if c["id"] not in dev_ids for it in d_items(c)]
dev_d = [it for c in d_cases if c["id"] in dev_ids for it in d_items(c)]
print(f"Signpost-D ({D_SET}): {len(d_cases)} cases | train {len(train_d)} questions | dev {len(dev_d)} questions "
      f"({len(dev_ids)} cases) | {dict(Counter(it['task'] for it in train_d))}")

Cloning into 'llms-the-hard-way'...


train routing  :  6319 | {'routing': 6319}
train questions:  9586 | {'noul': 5070, 'choice': 2596, 'score': 1920}
dev routing    :   531 | {'routing': 531}
dev questions  :  1014 | {'noul': 550, 'choice': 236, 'score': 228}
dev routing domains (unseen in training): ['campsite', 'hearing_clinic', 'home_cinema_installer', 'hr', 'motorbike_rental', 'museum', 'opticians']
Signpost-D (pilot): 300 cases | train 1113 questions | dev 279 questions (60 cases) | {'d_yes_no': 375, 'd_choice': 395, 'd_score': 343}


In [3]:
# Cell 2b — Signpost's original tests (the forgetting check)

real_convs = load("realistic/conversations.json")                                      # E10
ft = load("first_turn/messages.json")
e11 = [{**m, "agents": ft["companies"][m["company"]]} for m in ft["messages"]]          # E11, with each message's agents
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
ind_q, ind_r, ind_f = load("independent/questions.json")["items"], load("independent/routing.json"), load("independent/first_turn.json")["messages"]
ind2_q, ind2_r, ind2_f = (load("independent2/questions.json")["items"], load("independent2/routing.json"),
                          load("independent2/first_turn.json")["messages"])
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 9
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])         # the 6 test companies of 02, Cell 16

turns = lambda convs: sum(len(c["user_turns"]) for c in convs)
print(f"E10 {len(real_convs)} conversations / {turns(real_convs)} turns | E11 {len(e11)} | banking {len(banking)} | E12 {len(e12)}")
print(f"independent: questions {len(ind_q)} | routing {len(ind_r)} conversations / {turns(ind_r)} turns | first messages {len(ind_f)}")
print(f"independent 2: questions {len(ind2_q)} (up to {max(len(q['options']) for q in ind2_q)} options) | routing {len(ind2_r)} "
      f"conversations / {turns(ind2_r)} turns (up to {max(len(c['agents']) for c in ind2_r)} agents) | first messages {len(ind2_f)}")

E10 35 conversations / 193 turns | E11 100 | banking 225 | E12 170
independent: questions 300 | routing 40 conversations / 172 turns | first messages 200
independent 2: questions 200 (up to 50 options) | routing 12 conversations / 70 turns (up to 11 agents) | first messages 50


In [4]:
# Cell 3 — The model: Signpost's merged encoder + a new LoRA, Signpost's scoring head, scores for any number of options

from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer

START_DIR = Path(snapshot_download(START, revision=REV[START]))


class Model:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(START_DIR / "encoder")
        self.tok.padding_side, self.tok.truncation_side = "right", "left"       # if too long, the start of the input is cut
        enc = AutoModel.from_pretrained(START_DIR / "encoder", dtype=torch.float32).to(DEVICE)
        enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})   # recompute, don't store, activations
        self.H = enc.config.hidden_size
        self.enc = get_peft_model(enc, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                  target_modules=["Wqkv", "Wo", "Wi"]))
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)
        self.head.load_state_dict(torch.load(START_DIR / "head.pt", map_location="cpu")["state_dict"])   # Signpost's head

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    @staticmethod
    def text(item, name):
        """messages SEP current agent SEP question SEP candidate: Signpost's format (signpost.py)."""
        desc = item["options"][name]
        parts = [f"{m['role']}: {m['content']}" for m in item["messages"]]
        parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}",
                  f"candidate: {name}. {desc}" if desc else f"candidate: {name}"]
        return SEP.join(parts)

    def scores(self, items):
        """-> [items, most options] scores (missing options -1e4): one encoder pass over every option of every item."""
        sizes = torch.tensor([len(it["options"]) for it in items], device=DEVICE)
        texts = [self.text(it, name) for it in items for name in it["options"]]
        enc = self.tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        s = self.head((hidden.float() * m).sum(1) / m.sum(1)).squeeze(-1)
        real = torch.arange(int(sizes.max()), device=DEVICE) < sizes[:, None]
        out = torch.full(real.shape, -1e4, device=DEVICE)
        out[real] = s
        return out

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
# Cell 4a — Losses: soft labels for Signpost-D items (+ an expected-level term for score questions); Signpost's ordinal
# loss for its original items; metrics on the dev cases (accuracy against the label's top option, KL, Brier, ECE)

SCORE_EPS = 0.1


def soft_loss(scores, items):
    """Cross-entropy against the labellers' distribution (= KL + a constant). Score questions add
    SCORE_DIST_W x |expected level - target expected level| / (levels - 1)."""
    K = scores.shape[1]
    target = torch.tensor([[it["soft"][n] for n in it["options"]] + [0.0] * (K - len(it["options"])) for it in items],
                          device=scores.device)
    target = target / target.sum(1, keepdim=True)
    logp = F.log_softmax(scores, dim=1)
    loss = -(target * logp).sum(1).mean()
    is_score = torch.tensor([it["task"] == "d_score" for it in items], device=scores.device)
    if is_score.any():
        idx = torch.arange(K, device=scores.device, dtype=scores.dtype)
        n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
        gap = ((logp.exp() * idx).sum(1) - (target * idx).sum(1)).abs() / (n_levels - 1).clamp(min=1)
        loss = loss + SCORE_DIST_W * gap[is_score].mean()
    return loss


def ordinal_loss(scores, items, target):
    """Signpost's loss for its original items (03, Cell 4a)."""
    logp = F.log_softmax(scores, dim=1)
    K = scores.shape[1]
    is_score = torch.tensor([it["task"] == "score" for it in items], device=scores.device)
    level = torch.tensor([[it["levels"].index(n) for n in it["options"]] + [-9] * (K - len(it["options"])) if it["task"] == "score"
                          else [-9] * K for it in items], device=scores.device, dtype=scores.dtype)
    true = level.gather(1, target[:, None])
    nbr = ((level - true).abs() == 1) & is_score[:, None]
    n_nbr = nbr.sum(1, keepdim=True)
    eps = SCORE_EPS * (n_nbr > 0)
    soft = torch.where(nbr, eps / n_nbr.clamp(min=1), F.one_hot(target, K) * (1 - eps))
    loss = -(soft * logp).sum(1).mean()
    if not is_score.any():
        return loss
    n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
    expected = (logp.exp() * level.clamp(min=0)).sum(1)
    return loss + 0.5 * ((expected - true[:, 0]).abs() / (n_levels - 1).clamp(min=1))[is_score].mean()


def d_predictions(model, items, batch=16):
    """-> [(item, {option: probability})] for Signpost-D items."""
    model.train(False)
    out = []
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            p = torch.softmax(model.scores(chunk), dim=1).float().cpu()
            out += [(it, {n: p[i, j].item() for j, n in enumerate(it["options"])}) for i, it in enumerate(chunk)]
    return out


def d_metrics(preds, temperature=1.0):
    """The benchmark's four measures, against our own soft labels: accuracy (top option = the label's top option), KL from the
    label, Brier, ECE (10 bins). With a temperature, probabilities are softened as p ** (1 / T)."""
    rows = []
    for it, p in preds:
        q = {n: max(v, 1e-12) ** (1 / temperature) for n, v in p.items()}
        z = sum(q.values())
        q = {n: v / z for n, v in q.items()}
        g = it["soft"]
        ok = max(q, key=q.get) == max(g, key=g.get)
        rows.append({"task": it["task"], "ok": ok, "conf": max(q.values()),
                     "kl": sum(gv * math.log(gv / max(q[n], 1e-9)) for n, gv in g.items() if gv > 0),
                     "brier": sum((q[n] - g[n]) ** 2 for n in g)})
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    return {"all": summ(rows), **{t: summ([r for r in rows if r["task"] == t]) for t in sorted({r["task"] for r in rows})}}


def fit_temperature(preds):
    """Temperature that minimises KL on our own dev cases (never the benchmark)."""
    grid = [0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5]
    return min(grid, key=lambda t: d_metrics(preds, t)["all"]["kl"])

In [6]:
# Cell 4b — The training loop: Signpost-D batches every epoch (+ a fresh sample of Signpost's own items for new+old);
# after each epoch the dev cases are scored and the epoch with the lowest dev KL is kept

def batches_for_epoch(epoch):
    rng = random.Random(SEED + epoch)
    d = train_d[:]; rng.shuffle(d)
    out = [d[i:i + BATCH_NEW] for i in range(0, len(d), BATCH_NEW)]
    if DATA_MIX == "new+old":
        r = rng.sample(train_route, min(OLD_PER_EPOCH // 2, len(train_route)))
        c = rng.sample(train_choice, min(OLD_PER_EPOCH // 2, len(train_choice)))
        out += [r[i:i + BATCH_ROUTE] for i in range(0, len(r), BATCH_ROUTE)]
        out += [c[i:i + BATCH_CHOICE] for i in range(0, len(c), BATCH_CHOICE)]
    rng.shuffle(out)
    return out


def train_model(model):
    per_epoch = len(batches_for_epoch(1))
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * per_epoch
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0, history = (float("inf"), None, 0), 0, time.perf_counter(), []
    log(f"training: {per_epoch} batches per epoch, {total} steps, DATA_MIX {DATA_MIX}")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        run = {"new": [], "old": []}
        for items in batches_for_epoch(epoch):
            items = [dict(it) for it in items]
            for it in items:                                    # options shuffled, so position can't be learned
                names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            out = model.scores(items)
            if items[0]["task"].startswith("d_"):
                loss, kind = soft_loss(out, items), "new"
            else:
                target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
                loss, kind = ordinal_loss(out, items, target), "old"
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run[kind].append(loss.item())
            step += 1
            if step % 50 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  epoch {epoch} step {step}/{total} | loss new {statistics.mean(run['new'][-50:] or [0]):.4f} "
                    f"old {statistics.mean(run['old'][-50:] or [0]):.4f} | ~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
        dev = d_metrics(d_predictions(model, dev_d))["all"]
        history.append({"epoch": epoch, **dev})
        log(f"epoch {epoch} done | dev {dev}")
        if dev["kl"] < best[0]:
            best = (dev["kl"], {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best[2]} (dev KL {best[0]:.4f})")
    return {"train_minutes": minutes, "best_epoch": best[2], "history": history}

In [7]:
# Cell 5a — Signpost's test functions (from 03): timing, a table helper, routing and question tests

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def table(rows, key, cols, groups=None):
    """Print right / total per group (key(row) -> group) and for ALL rows; cols = {column: row -> True if right}."""
    groups = list(groups or sorted({key(r) for r in rows})) + ["ALL"]
    w = max(len(str(g_)) for g_ in groups)
    print(f"  {'':{w}s} | {'n':>4} | " + " | ".join(f"{c:>12s}" for c in cols))
    for g_ in groups:
        rs = [r for r in rows if g_ == "ALL" or key(r) == g_]
        if rs:
            print(f"  {g_:{w}s} | {len(rs):>4} | " + " | ".join(f"{sum(map(f_, rs)):>5} / {len(rs):<4}" for f_ in cols.values()))


def user(text):
    return [{"role": "user", "content": text}]


def live_routing(model, convs, given=False):
    rows, lat = [], []
    for conv in convs:
        cur = None
        for u in conv["user_turns"]:
            p, ms = timed_probs(model, {"messages": conv["messages"][:u["message_index"] + 1], "current": cur, "question": ROUTE_Q,
                                        "options": conv["agents"]})
            pick = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv.get("company"), "type": u["turn_type"], "flag": u.get("flag"), "ok": pick == u["expected_agent"]})
            lat.append(ms)
            cur = u["expected_agent"] if given else pick
    return rows, lat


def first_messages(model, msgs):
    rows, lat = [], []
    for m in msgs:
        p, ms = timed_probs(model, {"messages": user(m["message"]), "current": None, "question": ROUTE_Q, "options": m["agents"]})
        names, top = list(m["agents"]), p.topk(2).indices.tolist()
        rows.append({"type": m["type"], "ok": names[top[0]] == m["target_agent"], "top2": m["target_agent"] in [names[i] for i in top]})
        lat.append(ms)
    return rows, lat


def banking_routing(model):
    rows, lat = [], []
    for ex in banking:
        p, ms = timed_probs(model, {"messages": ex["messages"], "current": ex["current_agent"], "question": ROUTE_Q, "options": BANKING_AGENTS})
        rows.append({"ok": list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"]})
        lat.append(ms)
    return rows, lat


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": user(it["state"]), "current": None, "question": it["question"],
                                    "options": {o: "" for o in it["options"]}})
        top = int(p.argmax())
        rows.append({**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"], "conf": p.max().item(),
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [8]:
# Cell 5b — Signpost's tests, run on the trained model: R = {result: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}
right = {"right": lambda r: r["ok"]}


def n_ok(rows, key="ok"):
    return sum(r[key] for r in rows), len(rows)


def evaluate(model):
    model.train(False)
    timed_probs(model, {"messages": user("warm-up"), "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    R, LAT = {}, {}
    for name, convs in [("E10", real_convs), ("IND", ind_r), ("IND2", ind2_r)]:
        log(f"{name}: routing every turn")
        rows, LAT[f"{name} routing"] = live_routing(model, convs)
        R[f"{name} routing live"], R[f"{name} routing given"] = n_ok(rows), n_ok(live_routing(model, convs, given=True)[0])
        if name == "E10":
            R["E10 test companies"] = n_ok([r for r in rows if r["company"] in TEST_COMPANIES])
        print(f"{name} routing live, per turn type (given previous agent: {R[f'{name} routing given'][0]})")
        table(rows, lambda r: r["type"], right)
    for name, msgs in [("E11", e11), ("IND", ind_f), ("IND2", ind2_f)]:
        log(f"{name}: first messages")
        rows, LAT[f"{name} first"] = first_messages(model, msgs)
        R[f"{name} first top-1"], R[f"{name} first top-2"] = n_ok(rows), n_ok(rows, "top2")
        table(rows, lambda r: r["type"], {"top-1": lambda r: r["ok"], "top-2": lambda r: r["top2"]})
    log("banking")
    rows, LAT["banking"] = banking_routing(model)
    R["banking 50"], R["banking 225"] = n_ok(rows[:50]), n_ok(rows)
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        log(f"{name}: questions")
        rows, LAT[f"{name} questions"] = questions(model, items)
        R[f"{name} questions"] = n_ok(rows)
        R.update({f"{name} {TYPE[t]}": n_ok([r for r in rows if r["type"] == t]) for t in TYPE if any(r["type"] == t for r in rows)})
        table(rows, lambda r: TYPE[r["type"]], right, [TYPE[t] for t in TYPE])
        sc = [r for r in rows if r["type"] == "score"]
        print(f"  score within 1 level: {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)}")
        if name == "E12":
            table(rows, lambda r: r["category"], right)
            print("  mistakes:")
            for r in [r for r in rows if not r["ok"]]:
                print(f"    {r['id']} {r['type']:6s} | {r['state'][:44]:44s} | {r['question'][:36]:36s} | "
                      f"expected {r['answer']:14s} | got {r['pick']} ({r['conf']:.2f})")
        if name == "IND":
            R["IND questions unflagged"] = n_ok([r for r in rows if not r.get("flag")])
            print(f"  without flagged items {R['IND questions unflagged'][0]} / {R['IND questions unflagged'][1]} | claim-check yes/no "
                  f"(truth always yes) {n_ok([r for r in rows if r.get('flag') == 'claim_check'])[0]}")
    for name, L in LAT.items():
        L = lat_summary(L)
        print(f"latency {name:16s} p50 {L['p50']:4.0f} ms | p95 {L['p95']:4.0f} ms ({L['n']} questions)")
    return R, LAT

In [9]:
# Cell 5c — The benchmark, scored once at the end (general mode: never used for training or for any choice above).
# Scoring copied from the benchmark's bench/score_with_jobs.py (score_one, 10-bin ECE).

def bench_split(split):
    f = hf_hub_download(BENCH, f"all/{split}-00000-of-00001.parquet", repo_type="dataset", revision=REV[BENCH])
    rows = pd.read_parquet(f).to_dict("records")
    for r in rows:
        r["state"], r["questions"], r["gold"] = (json.loads(r[k]) for k in ("state", "questions", "gold"))
    return rows


def score_one(pred, gold):
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":
        pred = dict(pred)
        pred["probabilities"] = {{"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()}
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p, am_g = p.index(max(p)), g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def bench_item(case, q):
    """A benchmark question as a Signpost item, in the as-published format of notebook 06 (raw JSON state)."""
    crit = q.get("criteria") or {}
    if q["type"] == "choice":
        opts, keys = {k.replace("_", " "): v for k, v in crit.items()}, list(crit)
    elif q["type"] == "noul":
        opts, keys = {"yes": "", "no": ""}, ["true", "false"]
    else:
        parts = [c.partition(":") for c in crit]
        named = all(sep and 0 < len(n.split()) <= 3 for n, sep, _ in parts) and len({n.strip() for n, _, _ in parts}) == len(parts)
        opts = {n.strip(): d.strip() for n, _, d in parts} if named else {f"level {i + 1}": c for i, c in enumerate(crit)}
        keys = [str(i) for i in range(len(crit))]
    return {"messages": [{"role": "user", "content": json.dumps(case["state"], ensure_ascii=False)}], "current": None,
            "question": q["instructions"], "options": opts}, keys


def bench_score(model, cases, temperature=1.0):
    model.train(False)
    rows = []
    for case in cases:
        for qid, q in case["questions"].items():
            item, keys = bench_item(case, q)
            p = [max(v, 1e-12) ** (1 / temperature) for v in model.probs(item).tolist()]
            z = sum(p)
            ok, kl, brier, conf = score_one({"probabilities": {k: v / z for k, v in zip(keys, p)}}, case["gold"][qid])
            rows.append({"type": q["type"], "workflow": case["workflow"], "ok": ok, "kl": kl, "brier": brier, "conf": conf})
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl_from_gold": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    out = {"all": summ(rows)}
    for f in ("type", "workflow"):
        out.update({f"{f}={v}": summ([r for r in rows if r[f] == v]) for v in sorted({r[f] for r in rows})})
    return out

In [10]:
# Cell 6 — Before training (Signpost as published) on our dev cases, then train, then every measure

if AMP and torch.cuda.memory_allocated(DEVICE) > 1e9:
    raise RuntimeError("the GPU already holds an earlier model: restart the kernel (or start a new session) and run all cells "
                       "once, from the top")
random.seed(SEED); torch.manual_seed(SEED)
model = Model()
model.enc.print_trainable_parameters()
BEFORE = d_metrics(d_predictions(model, dev_d))
log(f"before training (Signpost as published), dev cases: {BEFORE['all']}")
trained = train_model(model)

dev_preds = d_predictions(model, dev_d)
T = fit_temperature(dev_preds)
AFTER = {"T=1": d_metrics(dev_preds), f"T={T}": d_metrics(dev_preds, T)}
for k, v in AFTER.items():
    log(f"after training, dev cases, {k}: " + " | ".join(f"{t} {m}" for t, m in v.items()))

R, LAT = evaluate(model)                                   # Signpost's original tests: how much was forgotten
TEST = bench_split("test")
BENCH_RES = {"T=1": bench_score(model, TEST), f"T={T}": bench_score(model, TEST, T)}
for k, v in BENCH_RES.items():
    log(f"benchmark test, {k}: {v['all']}")
    for g, m in v.items():
        if g != "all":
            print(f"  {g:38s} acc {m['accuracy']:.3f}  KL {m['kl_from_gold']:.3f}  Brier {m['brier']:.3f}  ECE {m['ece']:.3f}  (n {m['n']})")
print("benchmark reference points (accuracy): uniform 0.308 · majority ~0.47 · Signpost as published 0.4175 (06) · teacher 0.735")
json.dump({"data_mix": DATA_MIX, "d_set": D_SET, "revisions": REV, **trained, "dev_before": BEFORE, "dev_after": AFTER, "temperature": T,
           "signpost_tests": {k: list(v) for k, v in R.items()}, "latency": {k: lat_summary(v) for k, v in LAT.items()},
           "benchmark": BENCH_RES}, open(f"signpost_d_results_{DATA_MIX.replace('+', '_')}.json", "w"), indent=1)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

trainable params: 7,196,672 || all params: 401,978,368 || trainable%: 1.7903
[06:17:49] before training (Signpost as published), dev cases: {'n': 279, 'accuracy': 0.3978, 'kl': 0.7445, 'brier': 0.4765, 'ece': 0.1739}
[06:17:49] training: 279 batches per epoch, 1674 steps, DATA_MIX new


/tmp/ipykernel_117/2558530588.py:44: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


[06:18:53]   epoch 1 step 50/1674 | loss new 1.1939 old 0.0000 | ~34.3 min left | GPU 1.7 GB (peak 3.1)
[06:19:53]   epoch 1 step 100/1674 | loss new 1.1111 old 0.0000 | ~32.5 min left | GPU 1.8 GB (peak 3.1)
[06:20:57]   epoch 1 step 150/1674 | loss new 1.0201 old 0.0000 | ~31.7 min left | GPU 1.7 GB (peak 3.5)
[06:21:58]   epoch 1 step 200/1674 | loss new 1.0859 old 0.0000 | ~30.5 min left | GPU 1.8 GB (peak 3.5)
[06:23:01]   epoch 1 step 250/1674 | loss new 1.1796 old 0.0000 | ~29.6 min left | GPU 1.7 GB (peak 3.5)
[06:24:03] epoch 1 done | dev {'n': 279, 'accuracy': 0.4265, 'kl': 0.6374, 'brier': 0.4076, 'ece': 0.0949}
[06:24:28]   epoch 2 step 300/1674 | loss new 1.1683 old 0.0000 | ~30.5 min left | GPU 1.8 GB (peak 3.5)
[06:25:30]   epoch 2 step 350/1674 | loss new 1.1192 old 0.0000 | ~29.0 min left | GPU 1.8 GB (peak 3.5)
[06:26:29]   epoch 2 step 400/1674 | loss new 1.1228 old 0.0000 | ~27.6 min left | GPU 1.8 GB (peak 3.5)
[06:27:35]   epoch 2 step 450/1674 | loss new 1.1857 o

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

[07:04:21] benchmark test, T=1: {'n': 2000, 'accuracy': 0.3965, 'kl_from_gold': 0.7832, 'brier': 0.3326, 'ece': 0.1466}
  type=choice                            acc 0.303  KL 1.023  Brier 0.428  ECE 0.227  (n 600)
  type=noul                              acc 0.570  KL 0.235  Brier 0.196  ECE 0.037  (n 600)
  type=score                             acc 0.336  KL 1.014  Brier 0.364  ECE 0.224  (n 800)
  workflow=agent_trace_observability     acc 0.280  KL 1.682  Brier 0.540  ECE 0.384  (n 500)
  workflow=customer_service              acc 0.466  KL 0.476  Brier 0.250  ECE 0.114  (n 500)
  workflow=invoice_processing            acc 0.398  KL 0.671  Brier 0.387  ECE 0.097  (n 500)
  workflow=security_incidents            acc 0.442  KL 0.304  Brier 0.154  ECE 0.072  (n 500)
[07:04:21] benchmark test, T=1.25: {'n': 2000, 'accuracy': 0.3965, 'kl_from_gold': 0.6672, 'brier': 0.308, 'ece': 0.1153}
  type=choice                            acc 0.303  KL 0.876  Brier 0.392  ECE 0.196  (n 600)
  type

In [11]:
# Cell 7 — Save signpost-d/ in Signpost's format (encoder merged, head, config, signpost.py) and a zip; nothing is uploaded

OUT = Path(f"signpost-d-{DATA_MIX.replace('+', '_')}")
shutil.rmtree(OUT, ignore_errors=True)
model.enc.save_pretrained(OUT / "adapter")
merged = model.enc.merge_and_unload()
merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
model.tok.save_pretrained(OUT / "encoder")
torch.save({"state_dict": {k: v.float().cpu() for k, v in model.head.state_dict().items()}, "hidden": model.H}, OUT / "head.pt")
json.dump({"name": "Signpost-D", "start": START, "start_revision": REV[START], "sep": SEP, "prefix": "", "max_len": MAX_LEN,
           "route_question": ROUTE_Q, "temperature": T, "data_mix": DATA_MIX, "d_set": D_SET, "best_epoch": trained["best_epoch"],
           "lora_r": LORA_R, "lora_alpha": LORA_ALPHA}, open(OUT / "signpost_config.json", "w"), indent=1)
shutil.copy(START_DIR / "signpost.py", OUT / "signpost.py")
shutil.make_archive(str(OUT), "zip", root_dir=".", base_dir=str(OUT))
log(f"saved {OUT}/ and {OUT}.zip ({Path(str(OUT) + '.zip').stat().st_size / 1e9:.2f} GB) | download it if the result is worth keeping")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[07:04:56] saved signpost-d-new/ and signpost-d-new.zip (0.76 GB) | download it if the result is worth keeping


## Reading the results

- **Dev cases, before → after:** does the new data teach the skill on unseen cases of our own?
- **Signpost's tests:** compare with Signpost as published (03 / ledger). A drop means forgetting. Run 2 (`new+old`) should hold them.
- **Benchmark:** a general-mode measure (our data only; the benchmark never chose anything). Compare with 0.4175.